# Machine Learning and Natural Language Processing
## Esercizio 3. Pochi esempi nel prompt: zero-shot, one-shot, few-shot

**Master AIDA, Edizione XVI, 2026.** Giuliano Lorenzo, PhD Student in Economics, Statistics and Data Science

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/giulianolorenzo-PhD/aida-ml-nlp/blob/main/esercizio_3.ipynb)

**Il problema.** Il servizio clienti di una banca deve smistare i messaggi dei clienti per **motivo di contatto** (intent).
I motivi e i loro confini li decide la banca: dal solo nome non sempre si capiscono.

Un LLM può classificare i messaggi senza addestramento. Nel prompt mettiamo le istruzioni e, volendo, degli **esempi risolti**:

| Prompt | Esempi |
|:--|:--|
| **zero-shot** | nessuno: solo le istruzioni |
| **one-shot** | un messaggio per motivo |
| **few-shot** | alcuni messaggi per motivo (qui 5) |

I pesi del modello non cambiano: impara il compito dal contesto (**in-context learning**).

## Indice
0. Setup
1. I dati
2. Il prompt
3. Zero-shot, one-shot, few-shot
4. Dove aiutano gli esempi
5. Senza LLM: un classificatore addestrato
6. Cosa portare a casa

## 0. Setup

Non serve la GPU. Le risposte del modello, con il loro costo, sono in un file (cache) che il notebook scarica
dal repository del corso (`USE_COURSE_CACHE = True`): gira **senza chiave e senza costi**, e mostra comunque il costo vero.
Per nuove richieste serve la chiave OpenRouter nei **Secrets** di Colab (`OPENROUTER_API_KEY`), come nell'esercizio 2;
con `USE_COURSE_CACHE = False` tutte le richieste partono da zero (circa 22 centesimi di dollaro).

In [1]:
%pip install -q openai datasets

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os
import warnings
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from datasets import load_dataset
from datasets.utils.logging import disable_progress_bar
from huggingface_hub.utils import logging as hub_logging
from openai import OpenAI
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import make_pipeline

disable_progress_bar()                           # niente barre di avanzamento durante il download
hub_logging.set_verbosity_error()                # nasconde gli avvisi tecnici dell'Hugging Face Hub
warnings.filterwarnings("ignore", message="Unknown solver options")   # avviso innocuo tra versioni di scipy e scikit-learn
pio.templates.default = "plotly_white"
pd.set_option("display.max_colwidth", 100)
SAVE_DIR = Path("salvataggi")
SAVE_DIR.mkdir(exist_ok=True)

COURSE = "https://raw.githubusercontent.com/giulianolorenzo-PhD/aida-ml-nlp/main"     # il repository del corso: da qui la cache
USE_COURSE_CACHE = True        # True: le risposte già pronte del corso; False: tutte le richieste da zero (serve la chiave)
try:                                             # Colab: la chiave è nei Secrets
    from google.colab import userdata
    API_KEY = userdata.get("OPENROUTER_API_KEY")
except Exception:                                # altrove, o senza Secret: variabile d'ambiente, se c'è
    API_KEY = os.environ.get("OPENROUTER_API_KEY")
print("Chiave caricata:", bool(API_KEY))
if not API_KEY:
    print("Nessuna chiave: si usano solo le risposte salvate nella cache.")

Chiave caricata: True


## 1. I dati

**BANKING77** (PolyAI, Casanueva et al., 2020; licenza CC BY 4.0): messaggi di clienti di una banca online, in inglese, con 77 motivi.
Ne usiamo 10, quelli che gestirebbe un team dedicato alle **carte**.

In [3]:
REVISION = "1846984189f2bc0ac51c6fc3a600fa9e1714e0c6"            # versione fissa del dataset
INTENTS = ["card_arrival", "card_delivery_estimate", "order_physical_card", "get_physical_card", "activate_my_card",
           "getting_spare_card", "card_not_working", "card_swallowed", "lost_or_stolen_card", "compromised_card"]

data = load_dataset("PolyAI/banking77", revision=REVISION)
names = data["train"].features["label"].names                     # dal numero dell'etichetta al nome del motivo


def load(split):
    """I messaggi di uno split, con il nome del motivo, solo per i nostri 10 motivi."""
    df = data[split].to_pandas().assign(intent=lambda d: d["label"].map(names.__getitem__))
    return df[df["intent"].isin(INTENTS)].reset_index(drop=True)


train, test = load("train"), load("test")
print(f"Training: {len(train)} messaggi (da qui gli esempi). Test: {len(test)} messaggi, 40 per motivo.")
sample = train.groupby("intent", sort=False).sample(1, random_state=0)[["intent", "text"]]   # un messaggio per motivo
sample

Training: 1120 messaggi (da qui gli esempi). Test: 400 messaggi, 40 per motivo.


,intent,text
26,card_arrival,How do I track the card you sent me?
208,card_delivery_estimate,How long will it take to deliver to the US?
372,card_not_working,What's going on with my card? I've tried multiple times and it never goes through.
391,lost_or_stolen_card,"My card has been stolen. I reported it with the police, but I need to report it with you as well."
484,get_physical_card,do i have to wait for a physical card before i get my pin
584,compromised_card,What can I do if my card details where stolen from my car? I think they used my card to buy gas.
686,card_swallowed,An ATM machine didn't return my card.
822,getting_spare_card,"I would like to give another card to my daughter, how can I proceed?"
898,order_physical_card,Can I get a new card?
1013,activate_my_card,How do I activate my card to start using it?


Un nome che inganna: `get_physical_card` sembra "ottenere la carta fisica", ma raccoglie le domande sul **PIN**.

In [4]:
pin = train.loc[train["intent"] == "get_physical_card", "text"].str.contains(r"\bpins?\b", case=False)   # la parola PIN
print(f"Messaggi di get_physical_card che contengono 'PIN': {pin.sum()} su {len(pin)}")

Messaggi di get_physical_card che contengono 'PIN': 105 su 106


## 2. Il prompt

- **Istruzioni**: il compito e l'elenco dei 10 motivi.
- **Esempi**: coppie (messaggio, motivo) inserite come turni di conversazione già avvenuti, prima del messaggio da classificare.
- **Schema JSON** con i 10 motivi ammessi: la risposta è sempre uno di loro.

Il modello è **Qwen3 235B-A22B** (Alibaba, open-weight): forte, ma senza ragionamento (ai modelli di ragionamento
gli esempi servono meno). Lo abbiamo scelto con un pilota su dati di training, tra tre modelli economici.

In [5]:
MODEL = "qwen/qwen3-235b-a22b-2507"
INSTRUCTIONS = ("You route messages from customers of a bank. Return the intent of the message, "
                "choosing one of these intents: " + ", ".join(INTENTS) + ".")
SCHEMA = {"type": "object", "additionalProperties": False, "required": ["intent"],
          "properties": {"intent": {"type": "string", "enum": INTENTS}}}

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=API_KEY or "nessuna", timeout=90, max_retries=3)
cache_path = SAVE_DIR / ("risposte_banca.json" if USE_COURSE_CACHE else "risposte_banca_mie.json")   # le vostre in un file a parte
if USE_COURSE_CACHE and not cache_path.exists():                   # la cache del corso: le risposte già pronte
    urlretrieve(f"{COURSE}/materiale/risposte_banca.json", cache_path)
cache = json.loads(cache_path.read_text()) if cache_path.exists() else {}


def ask(text, examples):
    """Il motivo di un messaggio secondo il modello, con il costo della chiamata. examples: coppie (messaggio, motivo)."""
    key = json.dumps([MODEL, INSTRUCTIONS, examples, text])
    if key not in cache:
        if not API_KEY:
            raise RuntimeError("Questa richiesta non è nella cache: serve una chiave OpenRouter (vedi Setup).")
        shots = [turn for message, intent in examples                                 # gli esempi come conversazione
                 for turn in ({"role": "user", "content": message},
                              {"role": "assistant", "content": json.dumps({"intent": intent})})]
        response = client.chat.completions.create(
            model=MODEL, temperature=0, max_tokens=50,
            messages=[{"role": "system", "content": INSTRUCTIONS}, *shots, {"role": "user", "content": text}],
            response_format={"type": "json_schema", "json_schema": {"name": "risposta", "strict": True, "schema": SCHEMA}},
            extra_body={"usage": {"include": True}, "provider": {"require_parameters": True}})
        cost = getattr(response.usage, "cost", 0) or 0                                   # il costo, se il fornitore lo indica
        cache[key] = {"intent": json.loads(response.choices[0].message.content)["intent"], "costo": cost}
    return cache[key]


def classify(texts, examples=()):
    """Molti messaggi in parallelo (8 alla volta): i motivi previsti e il costo totale. Poi salva la cache."""
    with ThreadPoolExecutor(8) as pool:
        answers = list(pool.map(lambda t: ask(t, list(examples)), texts))
    cache_path.write_text(json.dumps(cache))
    return [a["intent"] for a in answers], sum(a["costo"] for a in answers)


print(ask("The ATM kept my card, what should I do?", []))

{'intent': 'card_swallowed', 'costo': 1.287e-05}


## 3. Zero-shot, one-shot, few-shot

Gli esempi vengono dal training; il test serve solo a misurare. Il risultato dipende anche da **quali** esempi
si mostrano: per one-shot e few-shot facciamo tre estrazioni a caso.
Misuriamo l'**accuratezza** (la quota di messaggi smistati bene) e la macro-F1.

In [6]:
def examples_for(k, draw):
    """k messaggi di training per motivo, in ordine casuale: le coppie (messaggio, motivo) da mostrare al modello."""
    chosen = train.groupby("intent").sample(k, random_state=draw).sample(frac=1, random_state=draw)
    return list(zip(chosen["text"], chosen["intent"]))


PROMPTS = {0: "zero-shot", 1: "one-shot", 5: "few-shot (5)"}
predictions, rows = {}, []
for k, prompt in PROMPTS.items():
    for draw in ([0] if k == 0 else [0, 1, 2]):                                    # zero-shot: nessun esempio da estrarre
        pred, cost = classify(test["text"], examples_for(k, draw) if k else [])
        predictions[k, draw] = pred
        rows.append({"Prompt": prompt, "Estrazione": draw + 1, "Accuratezza": accuracy_score(test["intent"], pred),
                     "Macro-F1": f1_score(test["intent"], pred, average="macro"),
                     "Costo per 1.000 messaggi ($)": 1000 * cost / len(test)})
results = pd.DataFrame(rows)

fig = px.strip(results, x="Prompt", y="Accuratezza", hover_data=["Estrazione"], range_y=[0, 1],
               title=f"Messaggi smistati bene: test, {len(test)} messaggi (un punto per estrazione degli esempi)")
fig.update_traces(marker_size=14)
fig.show()
results.round(3)

,Prompt,Estrazione,Accuratezza,Macro-F1,Costo per 1.000 messaggi ($)
0,zero-shot,1,0.710,0.689,0.014
1,one-shot,1,0.862,0.852,0.036
2,one-shot,2,0.862,0.857,0.039
3,one-shot,3,0.870,0.867,0.040
4,few-shot (5),1,0.910,0.908,0.136
5,few-shot (5),2,0.942,0.942,0.134
6,few-shot (5),3,0.932,0.933,0.143


- **Zero-shot** il modello conosce solo i nomi dei motivi e sbaglia quasi un messaggio su tre.
- **Un esempio per motivo** basta per un salto netto; **cinque** migliorano ancora. Quali esempi si estraggono
  cambia il risultato di qualche punto.
- Gli esempi si pagano **in ogni richiesta**: con cinque per motivo il costo è circa dieci volte quello zero-shot,
  sempre nell'ordine dei centesimi di dollaro per 1.000 messaggi.

## 4. Dove aiutano gli esempi

La F1 di ogni motivo, in media sulle tre estrazioni.

In [7]:
draws = {k: [d for kk, d in predictions if kk == k] for k in PROMPTS}
per_intent = pd.DataFrame({prompt: np.mean([f1_score(test["intent"], predictions[k, d], labels=INTENTS, average=None)
                                            for d in draws[k]], axis=0)
                           for k, prompt in PROMPTS.items()}, index=INTENTS)
fig = px.bar(per_intent.reset_index(names="Motivo").melt(id_vars="Motivo", var_name="Prompt", value_name="F1"),
             x="Motivo", y="F1", color="Prompt", barmode="group", title="F1 per motivo (test)")
fig.show()
per_intent.round(2)

,zero-shot,one-shot,few-shot (5)
card_arrival,0.66,0.81,0.87
card_delivery_estimate,0.63,0.70,0.82
order_physical_card,0.20,0.55,0.87
get_physical_card,0.05,0.76,0.91
activate_my_card,0.76,0.94,0.94
getting_spare_card,0.82,0.94,0.99
card_not_working,0.86,0.95,0.97
card_swallowed,0.99,1.00,1.00
lost_or_stolen_card,0.95,0.96,0.94
compromised_card,0.99,0.98,0.96


- Gli esempi servono dove il nome del motivo non basta: `get_physical_card` (il PIN) e `order_physical_card`.
- Dove il nome è chiaro (`card_swallowed`, `lost_or_stolen_card`, `compromised_card`) lo zero-shot va già bene
  e gli esempi aggiungono poco o nulla.

I messaggi sul PIN: le risposte zero-shot e few-shot (prima estrazione).

In [8]:
pin_rows = test.index[test["intent"] == "get_physical_card"][:5]
pd.DataFrame({"Messaggio": test.loc[pin_rows, "text"],
              "Zero-shot": [predictions[0, 0][i] for i in pin_rows],
              "Few-shot (5)": [predictions[5, 0][i] for i in pin_rows]})

,Messaggio,Zero-shot,Few-shot (5)
160,What do I do with my card PIN?,activate_my_card,get_physical_card
161,Is the PIN delivered separately?,card_delivery_estimate,get_physical_card
162,I do not have my pin,activate_my_card,get_physical_card
163,Are PIN separately?,card_arrival,get_physical_card
164,I cannot seem to find my PIN. Where is it?,activate_my_card,get_physical_card


Zero-shot il modello legge il nome alla lettera. Dove finiscono i messaggi dei due motivi:

In [9]:
where = pd.crosstab(test["intent"], pd.Series(predictions[0, 0], name="risposta zero-shot"))
where = where.loc[["get_physical_card", "order_physical_card"]]
where.loc[:, where.any()]                                                  # solo le colonne con almeno un messaggio

risposta zero-shot,activate_my_card,card_arrival,card_delivery_estimate,card_not_working,get_physical_card,order_physical_card
intent,,,,,,
get_physical_card,22,8,2,6,2,0
order_physical_card,0,1,7,1,26,5


- Le richieste di una nuova carta (`order_physical_card`) finiscono per lo più in `get_physical_card`:
  letto alla lettera, è il nome giusto.
- Le domande sul PIN finiscono soprattutto in `activate_my_card`.

## 5. Senza LLM: un classificatore addestrato

TF-IDF e regressione logistica (come nell'esercizio 1), addestrati sugli **stessi 5 esempi per motivo** del few-shot
e su **tutto il training**. Impostazioni di default, nessuna scelta.

In [10]:
def tfidf_f1(texts, labels):
    model = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True), LogisticRegression(max_iter=2000))
    return f1_score(test["intent"], model.fit(texts, labels).predict(test["text"]), average="macro")


few = [tfidf_f1(*zip(*examples_for(5, d))) for d in draws[5]]          # zip(*coppie): messaggi e motivi separati
llm = results.loc[results["Prompt"] == PROMPTS[5], "Macro-F1"]
baseline = pd.DataFrame({"Esempi etichettati": [50, 50, len(train)],
                         "Macro-F1 (media delle estrazioni)": [llm.mean(), np.mean(few), tfidf_f1(train["text"], train["intent"])]},
                        index=["LLM few-shot (5 per motivo)", "TF-IDF, stessi esempi", "TF-IDF, tutto il training"])
baseline.round(3)

,Esempi etichettati,Macro-F1 (media delle estrazioni)
LLM few-shot (5 per motivo),50,0.928
"TF-IDF, stessi esempi",50,0.606
"TF-IDF, tutto il training",1120,0.924


- Con gli **stessi 50 esempi** il classificatore addestrato resta molto indietro: l'LLM parte da ciò che sa della lingua,
  gli esempi gli mostrano le convenzioni della banca.
- Con **tutto il training** (oltre mille messaggi) il classificatore arriva allo stesso livello, e usarlo non costa quasi nulla.
  Il few-shot conviene quando gli esempi etichettati sono **pochi**.

In [11]:
summary = {"modello": MODEL, "frasi_test": len(test), "frasi_train": len(train), "campione": sample.to_dict("records"),
           "pin": {"con_pin": int(pin.sum()), "totale": len(pin)},
           "risultati": results.to_dict("records"), "per_motivo": per_intent.to_dict("index"),
           "pin_esempi": [{"testo": test.loc[i, "text"], "zero": predictions[0, 0][i], "few": predictions[5, 0][i]} for i in pin_rows],
           "zero_shot_dove": where.to_dict("index"),
           "tfidf": baseline.reset_index(names="Modello").to_dict("records")}
(SAVE_DIR / "risultati_esercizio_3.json").write_text(json.dumps(summary, indent=1))
print("Salvato:", SAVE_DIR / "risultati_esercizio_3.json")

Salvato: salvataggi/risultati_esercizio_3.json


## 6. Cosa portare a casa

1. **Zero-shot** il modello conosce solo i nomi delle classi; **pochi esempi** nel prompt gli insegnano le convenzioni
   del nostro compito, senza addestramento.
2. Gli esempi si pagano in ogni richiesta, e **quali** esempi si scelgono conta: si misura su più estrazioni.
3. Con pochi dati etichettati il few-shot batte un classificatore addestrato sugli stessi esempi;
   con molti dati, un classificatore classico è competitivo e costa molto meno.

**Per il project work**: per smistare ticket, e-mail o richieste servono poche decine di esempi etichettati e un prompt.
Misurate zero-shot e few-shot su un test, prima di etichettare migliaia di testi.